# B3 — Clasificador de Subvenciones

Notebook de clasificación taxonómica para publicaciones del dominio de subvenciones y ayudas públicas.
Sigue la misma arquitectura incremental que B0 (ambiental-energético), B1 (hídrico y natural) y B2 (urbanístico).

**Particularidades de B3 frente a bloques anteriores:**
- Las categorías N2 no son temáticas sino **fases del ciclo de vida** de la ayuda (bases → convocatoria → concesión → reintegro).
- La dimensión N3 es el **sector destinatario** (10 sectores).
- Multilabel estructural: ~430 registros del corpus aprueban bases Y convocan en el mismo acto.
- Frontera con B5 (empleo público): las "bases reguladoras" de bolsas de trabajo y procesos selectivos NO son subvenciones.

In [1]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())

True

In [2]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIChatModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

In [3]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")

OK: qwen/qwen3.5-9b listo  |  otros: ['gemma-4-e4b-it', 'google/gemma-4-e4b', 'google/gemma-3n-e4b', 'gemma-4-e2b-it', 'google/gemma-3-4b', 'text-embedding-nomic-embed-text-v1.5']


## 1. Schema B3

In [4]:
from clasificador.schema_B3 import (
    ActType, CategoryTypeB3, SubcategoryTypeB3, ClassifierOutputB3
)

In [5]:
ejemplo_valido = ClassifierOutputB3(
    is_relevant=True,
    act_type=ActType.ORDEN,
    categories=[CategoryTypeB3.SUB_BASE, CategoryTypeB3.SUB_CONV],
    subcategories=[SubcategoryTypeB3.SECTOR_INDUSTRIA],
    confidence=0.95,
    reasoning="'se establecen las bases reguladoras' → SUB_BASE; 'y se procede a su convocatoria' → SUB_CONV. 'ciberseguridad industrial' → sector_industria.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutputB3(
        is_relevant=False, act_type=ActType.RESOLUCION,
        categories=[CategoryTypeB3.SUB_CONV], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "orden",
  "categories": [
    "SUB_BASE",
    "SUB_CONV"
  ],
  "subcategories": [
    "sector_industria"
  ],
  "confidence": 0.95,
  "reasoning": "'se establecen las bases reguladoras' → SUB_BASE; 'y se procede a su convocatoria' → SUB_CONV. 'ciberseguridad industrial' → sector_industria."
}

Violación de invariante:
  ValidationError -> Value error, is_relevant=False con categories != []


## 2. Exploración del corpus B3

In [6]:
from clasificador.agent import get_ambito, inferir_act_type

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")

Corpus total: 65,201 registros | Columnas: ['id', 'pdf_link', 'expediente', 'promotor', 'proyecto', 'description', 'tipo', 'clean_id', 'bulletin', 'provincias', 'municipios', 'raw_scraped_timestamp', 'raw_scraped_year_month', 'scraped_timestamp', 'scraped_year_month', 'publication_timestamp', 'publication_type', 'contains_aau', 'proxy_pdf_link', 'ambito', 'rango']


In [7]:
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

keywords_dominio_B3 = [
    "extracto de la", "convocatoria de subvenc", "convocatoria de ayuda",
    "se convocan", "bdns", "concesión de subvenc", "concesión de ayuda",
    "concesión directa", "se conceden", "bases reguladoras",
    "reintegro de subvenc", "reintegro de ayuda", "pérdida del derecho al cobro",
]
desc_lower = df["description"].str.lower()
mask_b3 = desc_lower.str.contains("|".join(keywords_dominio_B3), na=False)
df_b3 = df[mask_b3].copy()
print(f"Universo B3 estimado: {len(df_b3):,} registros ({len(df_b3)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución por boletín (top 10):")
print(df_b3["bulletin"].value_counts().head(10).to_string())
print(f"\nDistribución N1 en universo B3:")
print(df_b3["act_type_n1"].value_counts().head(8).to_string())

Universo B3 estimado: 4,570 registros (7.01% del corpus)

Distribución por boletín (top 10):
bulletin
dogc    578
dog     514
boib    416
dogv    360
bopa    280
bon     272
doe     256
boc     251
boja    246
boe     231

Distribución N1 en universo B3:
act_type_n1
resolución            1760
extracto              1319
orden                  506
anuncio                399
corrección_errores     156
convocatoria           128
edicto                  68
notificación            67


## 3. Ground truth — Muestreo estratificado

Cuotas por grupo (total: 150):

| Grupo | Cuota | Notas |
|-------|-------|-------|
| SUB_CONV | 40 | extractos BDNS + convocatorias |
| SUB_CON | 30 | concesiones y propuestas de resolución |
| SUB_BASE | 25 | bases sin señal de convocatoria ni RRHH |
| SUB_REV | 10 | pool muy pequeño (~10), se toman todos |
| MULTILABEL_BC | 15 | bases + convocatoria en el mismo acto |
| NEGATIVO_RRHH | 15 | bases reguladoras de bolsas/procesos selectivos (frontera B5) |
| NEGATIVO | 15 | resto del corpus sin señal B3 |

In [8]:
keywords_B3 = {
    "SUB_CONV": ["extracto de la", "convocatoria de subvenc", "convocatoria de ayuda",
                 "se convocan subvenc", "se convocan ayuda", "bdns"],
    "SUB_CON":  ["concesión de subvenc", "concesión de ayuda", "concesión directa de",
                 "se conceden"],
    "SUB_BASE": ["bases reguladoras"],
    "SUB_REV":  ["reintegro de subvenc", "reintegro de ayuda",
                 "pérdida del derecho al cobro", "revocación de subvenc"],
}

cuotas_B3 = {"SUB_CONV": 40, "SUB_CON": 30, "SUB_BASE": 25, "SUB_REV": 10}
N_MULTILABEL_BC = 15
N_NEGATIVO_RRHH = 15
N_NEGATIVOS     = 15

# Señales de empleo público para detectar bases reguladoras de RRHH (no subvención)
kws_rrhh = ["bolsa de", "proceso selectivo", "pruebas selectivas", "concurso-oposición",
            "concurso oposición", "personal laboral", "funcionari", "plazas de"]

desc_lower = df["description"].str.lower()
mask_rrhh = desc_lower.str.contains("|".join(kws_rrhh), na=False)

print(f"{'Label':<14} {'Pool':>8} {'Cuota':>7} {'Estado':>14}")
print("-" * 48)
for label, kws in keywords_B3.items():
    mask = desc_lower.str.contains("|".join(kws), na=False)
    if label == "SUB_BASE":
        mask = mask & ~mask_rrhh  # bases de RRHH van a su propio grupo
    pool = mask.sum()
    cuota = cuotas_B3.get(label, 0)
    estado = "OK" if pool >= cuota else f"REDUCIDA a {min(cuota, pool)}"
    print(f"{label:<14} {pool:>8,} {cuota:>7} {estado:>14}")

mask_bc = (desc_lower.str.contains("bases reguladoras", na=False)
           & desc_lower.str.contains("convoca", na=False) & ~mask_rrhh)
mask_neg_rrhh = desc_lower.str.contains("bases reguladoras", na=False) & mask_rrhh
print(f"{'MULTILABEL_BC':<14} {mask_bc.sum():>8,} {N_MULTILABEL_BC:>7}")
print(f"{'NEGATIVO_RRHH':<14} {mask_neg_rrhh.sum():>8,} {N_NEGATIVO_RRHH:>7}")

Label              Pool   Cuota         Estado
------------------------------------------------
SUB_CONV          2,243      40             OK
SUB_CON           1,211      30             OK
SUB_BASE          1,177      25             OK
SUB_REV              10      10             OK
MULTILABEL_BC       775      15
NEGATIVO_RRHH       252      15


In [9]:
sampled_ids = set()
frames = []

def tomar_muestra(pool_df, n, grupo):
    n = min(n, len(pool_df))
    if n == 0:
        print(f"  {grupo:<14} SKIP (pool vacío)")
        return
    sample = pool_df.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<14} pool={len(pool_df):>5,}  sampled={n}")

# 1. SUB_REV primero (pool minúsculo, prioridad máxima)
mask = desc_lower.str.contains("|".join(keywords_B3["SUB_REV"]), na=False) & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], cuotas_B3["SUB_REV"], "SUB_REV")

# 2. Multilabel bases+convocatoria
mask = mask_bc & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], N_MULTILABEL_BC, "MULTILABEL_BC")

# 3. Negativos difíciles: bases reguladoras de RRHH
mask = mask_neg_rrhh & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], N_NEGATIVO_RRHH, "NEGATIVO_RRHH")

# 4. Categorías principales
for label in ["SUB_CONV", "SUB_CON", "SUB_BASE"]:
    mask = desc_lower.str.contains("|".join(keywords_B3[label]), na=False)
    if label == "SUB_BASE":
        mask = mask & ~mask_rrhh
    mask = mask & ~df.index.isin(sampled_ids)
    tomar_muestra(df[mask], cuotas_B3[label], label)

# 5. Negativos del resto del corpus
all_kws_b3 = [kw for kws in keywords_B3.values() for kw in kws]
mask_neg = (~desc_lower.str.contains("|".join(all_kws_b3), na=False)
            & ~df.index.isin(sampled_ids))
tomar_muestra(df[mask_neg], N_NEGATIVOS, "NEGATIVO")

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")
print(df_muestreo["grupo_muestreo"].value_counts().to_string())

  SUB_REV        pool=   10  sampled=10
  MULTILABEL_BC  pool=  775  sampled=15
  NEGATIVO_RRHH  pool=  252  sampled=15
  SUB_CONV       pool=2,234  sampled=40
  SUB_CON        pool=1,204  sampled=30
  SUB_BASE       pool=1,152  sampled=25
  NEGATIVO       pool=61,255  sampled=15

Total muestreado: 150 registros
grupo_muestreo
SUB_CONV         40
SUB_CON          30
SUB_BASE         25
MULTILABEL_BC    15
NEGATIVO_RRHH    15
NEGATIVO         15
SUB_REV          10


In [10]:
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul  = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()

Validación del muestreo - 3 ejemplos por grupo:

--- SUB_REV ---
  [DOCM] Notificación de 24/01/2025, de la Delegación Provincial de Educación, Cultura y Deportes de Albacete, de acuerdos de inicio de rei
  [BOCM] Concesión subvenciones – Orden de 27 de diciembre de 2024, de la Consejería de Vivienda, Transportes e Infraestructuras, por la qu
  [BOJA] Acuerdo de 25 de febrero de 2025, de la Delegación Territorial de Fomento, Articulación del Territorio y Vivienda en Málaga, por e

--- MULTILABEL_BC ---
  [DOG] RESOLUCIÓN de 9 de diciembre de 2024 por la que se da publicidad del acuerdo del Consejo de Dirección que aprueba las bases regula
  [DOG] ORDEN de 30 de diciembre de 2024 por la que se establecen las bases reguladoras para la concesión de subvenciones para la prevenci
  [BOJA] Extracto de 4 de febrero de 2025, de la Universidad de Huelva, por la que se aprueban las bases reguladoras y la convocatoria de P

--- NEGATIVO_RRHH ---
  [DOGC] Anuncio sobre publicación de las bases reg

In [11]:
PATH_MUESTREO = "../data/ground_truth/ground_truth_B3_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")

Guardado: ../data/ground_truth/ground_truth_B3_muestreo.csv  (150 registros)
Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt


## 4. Agente base

In [12]:
from clasificador.schema_B3 import ClassifierOutputB3
from clasificador.prompts_B3 import PROMPT_REGISTRY_B3
from clasificador.agent import build_agent, run_experiment

In [13]:
agent_b3_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB3,
    prompt_registry=PROMPT_REGISTRY_B3,
)

# Casos cualitativos representativos del dominio B3
casos_b3 = [
    ("EXTRACTO de la Orden de 6 de marzo de 2025, de la Consejería de Industria, Comercio y Empleo, por la que se convocan subvenciones para el año 2025 dirigidas a impulsar la ciberseguridad industrial en Castilla y León.", "bocyl"),
    ("ORDEN de 23 de diciembre de 2024 por la que se establecen las bases reguladoras de los premios a la cooperación y se procede a su convocatoria.", "dog"),
    ("Propuesta de resolución provisional del director general de Energía y Cambio Climático por la que se conceden subvenciones para la realización de instalaciones de autoconsumo con fuentes de energía renovable.", "boib"),
    ("Edicto sobre aprobación de las bases reguladoras y la convocatoria que regirán el proceso selectivo, mediante concurso oposición por promoción interna, para cubrir dos plazas de técnico medio.", "dogc"),
    ("Notificación de 06/02/2025, de la Delegación Provincial de Educación, Cultura y Deportes de Toledo, de acuerdos de inicio de expedientes de reintegro de ayuda al estudio.", "docm"),
]

for desc, bul in casos_b3:
    print(f"\n[{bul.upper()}] {desc[:80]}...")
    # result = await agent_b3_v1.run(f"Boletín: {bul.upper()}\n\nDescripción: {desc}")
    # print(result.output.model_dump_json(indent=2))


[BOCYL] EXTRACTO de la Orden de 6 de marzo de 2025, de la Consejería de Industria, Comer...

[DOG] ORDEN de 23 de diciembre de 2024 por la que se establecen las bases reguladoras ...

[BOIB] Propuesta de resolución provisional del director general de Energía y Cambio Cli...

[DOGC] Edicto sobre aprobación de las bases reguladoras y la convocatoria que regirán e...

[DOCM] Notificación de 06/02/2025, de la Delegación Provincial de Educación, Cultura y ...


## 5. Funciones de evaluación B3

In [14]:
def parse_labels(value) -> set:
    """Convierte cualquier representación de etiquetas a un set de strings."""
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    s = str(value).strip()
    if s.startswith("["):
        try:
            items = json.loads(s)
            return {str(i).strip('"') for i in items if i}
        except json.JSONDecodeError:
            pass
    return {v.strip().strip('"') for v in s.split(",") if v.strip()}

In [15]:
def compute_metrics_B3(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B3.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    ALL_CATS_B3 = [e.value for e in CategoryTypeB3]
    mlb = MultiLabelBinarizer(classes=ALL_CATS_B3)
    mlb.fit([ALL_CATS_B3])

    gt_labels   = [parse_labels(v) & set(ALL_CATS_B3) for v in df_eval["categories_gt"]]
    pred_labels = [parse_labels(v) & set(ALL_CATS_B3) for v in df_eval["categories_pred"]]
    Y    = mlb.transform(gt_labels)
    Yhat = mlb.transform(pred_labels)

    is_rel_gt   = df_eval["is_relevant_gt"].astype(bool)
    is_rel_pred = df_eval["is_relevant_pred"].astype(bool)

    exact = pd.Series([set(g) == set(p) for g, p in zip(gt_labels, pred_labels)])
    rel   = is_rel_gt

    metrics = {
        "is_rel_accuracy":  round(accuracy_score(is_rel_gt, is_rel_pred), 4),
        "is_rel_precision": round(precision_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_recall":    round(recall_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_f1":        round(f1_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "micro_f1":         round(f1_score(Y, Yhat, average="micro", zero_division=0), 4),
        "macro_f1":         round(f1_score(Y, Yhat, average="macro", zero_division=0), 4),
        "hamming_loss":     round(hamming_loss(Y, Yhat), 4),
        "jaccard_samples":  round(jaccard_score(Y, Yhat, average="samples", zero_division=0), 4),
        "subset_accuracy":  round(accuracy_score(Y, Yhat), 4),
        "exact": exact,
        "rel":   rel,
    }

    if verbose:
        title = f"-- {label} --" if label else "-- Métricas B3 --"
        print(f"\n{title}\n")
        tp = int((is_rel_gt & is_rel_pred).sum())
        fp = int((~is_rel_gt & is_rel_pred).sum())
        fn = int((is_rel_gt & ~is_rel_pred).sum())
        tn = int((~is_rel_gt & ~is_rel_pred).sum())
        print(f"is_relevant  Acc={metrics['is_rel_accuracy']:.3f}  P={metrics['is_rel_precision']:.3f}  "
              f"R={metrics['is_rel_recall']:.3f}  F1={metrics['is_rel_f1']:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {metrics['micro_f1']:.3f}")
        print(f"  Macro F1:      {metrics['macro_f1']:.3f}")
        print(f"  Hamming Loss:  {metrics['hamming_loss']:.4f}")
        print(f"  Jaccard:       {metrics['jaccard_samples']:.3f}")
        print(f"  Subset Acc:    {metrics['subset_accuracy']:.3f}\n")
        f1s = f1_score(Y, Yhat, average=None, zero_division=0)
        sups = Y.sum(axis=0)
        print(f"  {'Label':<12} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print(f"  {'-'*37}")
        for i, cat in enumerate(ALL_CATS_B3):
            prec = precision_score(Y[:, i], Yhat[:, i], zero_division=0)
            rec  = recall_score(Y[:, i], Yhat[:, i], zero_division=0)
            print(f"  {cat:<12} {prec:>6.3f} {rec:>6.3f} {f1s[i]:>6.3f} {int(sups[i]):>5}")
        print(f"  {'-'*37}")
        print(f"  {'Macro':<12} {'':>6} {'':>6} {metrics['macro_f1']:>6.3f}\n")

        cards = [len(g) for g in gt_labels]
        print(f"  Subset Acc por cardinalidad:")
        for card in [0, 1, 2]:
            idx = [i for i, c in enumerate(cards) if c == card]
            if idx:
                acc = accuracy_score(Y[idx], Yhat[idx])
                print(f"    card={card} ({'no relevante' if card == 0 else str(card)}) : {acc:.3f}  ({int(acc*len(idx))}/{len(idx)})")
        idx3 = [i for i, c in enumerate(cards) if c >= 3]
        if idx3:
            acc3 = accuracy_score(Y[idx3], Yhat[idx3])
            print(f"    card>=3               : {acc3:.3f}  ({int(acc3*len(idx3))}/{len(idx3)})")

        conf = df_eval.get("confidence", pd.Series(dtype=float))
        if conf.notna().any():
            print(f"\n  Confianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return metrics

In [16]:
def print_errors_B3(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    lbl = f" · {label}" if label else ""
    print(f"\n-- Errores N2 en relevantes{lbl} --")
    print(f"Total: {len(errores)}\n")
    for _, row in errores.iterrows():
        gt   = sorted(parse_labels(row["categories_gt"]))
        pred = sorted(parse_labels(row["categories_pred"]))
        falta = sorted(set(gt) - set(pred))
        sobra = sorted(set(pred) - set(gt))
        desc  = str(row["description"])[:90].replace("\n", " ")
        razon = str(row.get("reasoning", "")).replace("\n", " ")[:120]
        print(f"ID {row['id']} | GT={gt} | PRED={pred}")
        print(f"  Falta: {falta} | Sobra: {sobra}")
        print(f"  {desc}...")
        print(f"  Razonamiento: {razon}")
        print()

---

##  6. Experimento 1 - Baseline zero-shot

**Problema**: No existe un clasificador para el dominio de subvenciones. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: Establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistemáticos que guiarán la mejora del prompt. Atención especial a: (1) frontera SUB_BASE/SUB_CONV en órdenes que aprueban y convocan a la vez, (2) frontera con empleo público en "bases reguladoras", (3) SUB_REV con soporte muy bajo.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B3_V1 · zero-shot · sin contexto N1.

**Resultados**: is_rel F1=0.982 · Micro F1=0.786 · **Macro F1=0.759** · Subset Acc=0.680 · 62.3 s/item.

| Label | P | R | F1 | Sup |
|-------|------|------|------|-----|
| SUB_CONV | 0.886 | 0.972 | 0.927 | 72 |
| SUB_CON | 0.889 | 0.533 | 0.667 | 15 |
| SUB_BASE | 0.387 | 0.960 | **0.552** | 25 |
| SUB_REV | 1.000 | 0.800 | 0.889 | 10 |

**Análisis de errores (47 en relevantes)**:
1. **SUB_BASE espuria (~30 errores, P=0.387)**: el modelo añade SUB_BASE a los EXTRACTOS cuyo acto extractado establece bases ("se establecen las bases... y se convocan" → predice [BASE, CONV] cuando el GT es [CONV]), e incluso la alucina en convocatorias simples que solo citan las bases como amparo (ids 42, 45, 59, 124).
2. **SUB_CON perdida (7 errores, R=0.533)**: decretos de concesión directa con "normas especiales reguladoras" → predice BASE (ids 88, 102); "resolución de concesión al amparo de" → predice CONV (ids 123, 126); publicación de concedidas → predice BASE/CONV (ids 127, 133).
3. **Relevancia casi perfecta** (F1=0.982): la frontera RRHH funciona. Los 3 FN: derogación de bases (129), distribución de créditos (70) y una notificación de reintegro (5).

Conclusión: el modelo etiqueta el acto EXTRACTADO en lugar del acto PUBLICADO. V2 ataca esto con la regla EXTRACTO reforzada y la regla "las referencias no etiquetan".

In [17]:
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B3_anotado.csv")
df_anotado_run = df_anotado[df_anotado["is_relevant_gt"].notna()].copy()

df_b3_exp1 = await run_experiment(
    agent_b3_v1, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b3_exp1_baseline_qwen9b.csv",
    desc="B3 Exp1 - Baseline V1",
)
df_b3_exp1.head(3)

  Reanudando: 150/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,docm,"Notificación de 24/01/2025, de la Delegación P...",SUB_REV,True,SUB_REV,sector_educacion,Inicio reintegro ayuda al estudio,True,notificación,"[""SUB_REV""]","[""sector_educacion""]",0.95,"El texto menciona explícitamente ""reintegro de...",108.660
1,1,bocm,Concesión subvenciones\n– Orden de 27 de dicie...,SUB_REV,True,SUB_REV,sector_vivienda,Perdida derecho al cobro subvenciones alquiler,True,orden,"[""SUB_REV""]","[""sector_vivienda""]",0.95,"El texto menciona ""pérdida del derecho al cobr...",63.623
2,2,boja,"Acuerdo de 25 de febrero de 2025, de la Delega...",SUB_REV,True,SUB_REV,sector_vivienda,Inicio perdida derecho al cobro ayudas alquiler,True,acuerdo,"[""SUB_REV""]","[""sector_vivienda""]",0.95,"El texto menciona ""declaración de pérdida del ...",66.729


In [18]:
df_b3_exp1 = pd.read_csv("../results/b3_exp1_baseline_qwen9b.csv")
df_eval_b3_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b3_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b3_1 = compute_metrics_B3(df_eval_b3_1, "Experimento B3-1 - Baseline")
print_errors_B3(df_eval_b3_1, m_b3_1["exact"], m_b3_1["rel"], label="B3 Experimento 1 - Baseline")


-- Experimento B3-1 - Baseline --

is_relevant  Acc=0.973  P=0.991  R=0.973  F1=0.982
             TP=107  FP=1  FN=3  TN=39

N2 multilabel:
  Micro F1:      0.786
  Macro F1:      0.759
  Hamming Loss:  0.1000
  Jaccard:       0.537
  Subset Acc:    0.680

  Label             P      R     F1   Sup
  -------------------------------------
  SUB_CONV      0.886  0.972  0.927    72
  SUB_CON       0.889  0.533  0.667    15
  SUB_BASE      0.387  0.960  0.552    25
  SUB_REV       1.000  0.800  0.889    10
  -------------------------------------
  Macro                       0.759

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.975  (39/40)
    card=1 (1) : 0.520  (51/98)
    card=2 (2) : 1.000  (12/12)

  Confianza: media=0.965  min=0.850  max=1.000

-- Errores N2 en relevantes · B3 Experimento 1 - Baseline --
Total: 47

ID 5 | GT=['SUB_REV'] | PRED=[]
  Falta: ['SUB_REV'] | Sobra: []
  Notificación de 06/02/2025, de la Delegación Provincial de Educación, Cultura y Deportes

---

##  7. Experimento 2 - Prompt v2 (correcciones post-baseline)

**Problema**: El baseline (Macro F1=0.759) etiqueta el acto extractado en lugar del acto publicado: añade SUB_BASE a los extractos BDNS y a convocatorias que solo citan las bases como amparo (P de SUB_BASE=0.387), y pierde SUB_CON en concesiones directas por decreto y publicaciones de concedidas (R de SUB_CON=0.533).

**Objetivo**: Verificar si las 5 reglas nuevas de V2 corrigen los errores: (1) regla EXTRACTO reforzada (extracto = solo SUB_CONV), (2) "las referencias no etiquetan" (al amparo de / reguladas en), (3) SUB_BASE exige acto completo cuyo objeto sean las bases, (4) concesión directa por decreto = SUB_CON, (5) notificaciones de reintegro = SUB_REV.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B3_V2 · zero-shot · sin contexto N1.

**Resultados**: is_rel F1=0.986 · Micro F1=0.923 · **Macro F1=0.891** (+0.132 vs baseline) · Subset Acc=0.907 (+0.227) · 63.6 s/item.

| Label | P | R | F1 | Δ F1 vs Exp1 |
|-------|------|------|------|------|
| SUB_CONV | 0.910 | 0.986 | 0.947 | +0.020 |
| SUB_CON | 0.923 | 0.800 | 0.857 | +0.190 |
| SUB_BASE | 0.920 | 0.920 | 0.920 | +0.368 |
| SUB_REV | 0.889 | 0.800 | 0.842 | -0.047 |

**Análisis de los 13 errores residuales**:
1. **Extracto que aún cuela SUB_BASE** (ids 14, 19): el razonamiento cita la REGLA EXTRACTO pero no la aplica. Necesita ejemplo few-shot.
2. **"Bases para la concesión" añade etiquetas espurias** (ids 93, 94, 124): la finalidad "para la concesión de subvenciones" arrastra SUB_CONV o SUB_CON.
3. **Publicación de concedidas → SUB_CONV** (ids 127, 133): la referencia larga "...y se convocan" sigue arrastrando.
4. **SUB_REV**: "pérdida del derecho al cobro" con encabezado "Concesión subvenciones" → SUB_CON (id 6); derogación de bases → SUB_REV espuria (id 129).
5. Menores: corrección de resolución que resuelve convocatoria → CONV (id 101), acto completo bases+convocatoria con BASE perdida (id 107), cadena de herencia doble (id 51).

V3 ataca 1-4 con tres reglas nuevas (finalidad, resolver convocatoria = SUB_CON, derogar ≠ SUB_REV) y 4 ejemplos few-shot quirúrgicos.

In [19]:
agent_b3_v2 = build_agent(
    model, "v2",
    output_type=ClassifierOutputB3,
    prompt_registry=PROMPT_REGISTRY_B3,
)

df_b3_exp2 = await run_experiment(
    agent_b3_v2, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b3_exp2_promptv2_qwen9b.csv",
    desc="B3 Exp2 - Prompt v2",
)
df_b3_exp2.head(3)

  Reanudando: 150/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,docm,"Notificación de 24/01/2025, de la Delegación P...",SUB_REV,True,SUB_REV,sector_educacion,Inicio reintegro ayuda al estudio,True,notificación,"[""SUB_REV""]","[""sector_educacion""]",0.95,"La descripción menciona ""reintegro de ayuda al...",109.122
1,1,bocm,Concesión subvenciones\n– Orden de 27 de dicie...,SUB_REV,True,SUB_REV,sector_vivienda,Perdida derecho al cobro subvenciones alquiler,True,orden,"[""SUB_REV""]","[""sector_vivienda""]",0.95,"La descripción contiene ""se declara la pérdida...",62.453
2,2,boja,"Acuerdo de 25 de febrero de 2025, de la Delega...",SUB_REV,True,SUB_REV,sector_vivienda,Inicio perdida derecho al cobro ayudas alquiler,True,acuerdo,"[""SUB_REV""]","[""sector_vivienda""]",0.95,"El texto menciona explícitamente la ""declaraci...",74.261


In [20]:
df_b3_exp2 = pd.read_csv("../results/b3_exp2_promptv2_qwen9b.csv")
df_eval_b3_2 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b3_exp2[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b3_2 = compute_metrics_B3(df_eval_b3_2, "Experimento B3-2 - Prompt v2")
print_errors_B3(df_eval_b3_2, m_b3_2["exact"], m_b3_2["rel"], label="B3 Experimento 2 - Prompt v2")


-- Experimento B3-2 - Prompt v2 --

is_relevant  Acc=0.980  P=0.991  R=0.982  F1=0.986
             TP=108  FP=1  FN=2  TN=39

N2 multilabel:
  Micro F1:      0.923
  Macro F1:      0.891
  Hamming Loss:  0.0317
  Jaccard:       0.667
  Subset Acc:    0.907

  Label             P      R     F1   Sup
  -------------------------------------
  SUB_CONV      0.910  0.986  0.947    72
  SUB_CON       0.923  0.800  0.857    15
  SUB_BASE      0.920  0.920  0.920    25
  SUB_REV       0.889  0.800  0.842    10
  -------------------------------------
  Macro                       0.891

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.975  (39/40)
    card=1 (1) : 0.878  (86/98)
    card=2 (2) : 0.917  (11/12)

  Confianza: media=0.956  min=0.750  max=1.000

-- Errores N2 en relevantes · B3 Experimento 2 - Prompt v2 --
Total: 13

ID 6 | GT=['SUB_REV'] | PRED=['SUB_CON']
  Falta: ['SUB_REV'] | Sobra: ['SUB_CON']
  Concesión subvenciones – Orden de 27 de diciembre de 2024, de la Con

---

##  8. Experimento 3 - Prompt v3 (few-shot)

**Problema**: V2 (Macro F1=0.891) deja 13 errores residuales con patrones identificados: 2 extractos que aún cuelan SUB_BASE pese a citar la regla, 3 casos donde la finalidad "bases para la concesión" añade etiquetas espurias, 2 publicaciones de concedidas arrastradas a SUB_CONV y 2 confusiones de SUB_REV.

**Objetivo**: Verificar si 3 reglas nuevas (la finalidad no etiqueta, resolver convocatoria = SUB_CON, derogar bases ≠ SUB_REV) y 4 ejemplos few-shot quirúrgicos (extracto con bases, publicación de concedidas, pérdida del derecho al cobro, bases para la concesión) eliminan los errores residuales.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B3_V3 (V2 + reglas 9-11 + 4 ejemplos, 6.361 chars) · few-shot · sin contexto N1.

**Resultados**: pendiente.

> Nota: para ejecutar tras actualizar prompts_B3.py hay que reiniciar el kernel (el import de PROMPT_REGISTRY_B3 no se recarga solo).

In [21]:
agent_b3_v3 = build_agent(
    model, "v3",
    output_type=ClassifierOutputB3,
    prompt_registry=PROMPT_REGISTRY_B3,
)

df_b3_exp3 = await run_experiment(
    agent_b3_v3, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b3_exp3_promptv3_qwen9b.csv",
    desc="B3 Exp3 - Prompt v3",
)
df_b3_exp3.head(3)

  Reanudando: 150/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,docm,"Notificación de 24/01/2025, de la Delegación P...",SUB_REV,True,SUB_REV,sector_educacion,Inicio reintegro ayuda al estudio,True,notificación,"[""SUB_REV""]","[""sector_educacion""]",0.95,"El texto contiene ""Notificación"" y ""inicio de ...",77.051
1,1,bocm,Concesión subvenciones\n– Orden de 27 de dicie...,SUB_REV,True,SUB_REV,sector_vivienda,Perdida derecho al cobro subvenciones alquiler,True,orden,"[""SUB_REV""]","[""sector_vivienda""]",0.98,"La publicación se refiere a una Orden que ""dec...",61.140
2,2,boja,"Acuerdo de 25 de febrero de 2025, de la Delega...",SUB_REV,True,SUB_REV,sector_vivienda,Inicio perdida derecho al cobro ayudas alquiler,True,acuerdo,"[""SUB_REV""]","[""sector_vivienda""]",0.95,"La descripción incluye explícitamente ""declara...",66.333


In [22]:
df_b3_exp3 = pd.read_csv("../results/b3_exp3_promptv3_qwen9b.csv")
df_eval_b3_3 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b3_exp3[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b3_3 = compute_metrics_B3(df_eval_b3_3, "Experimento B3-3 - Prompt v3")
print_errors_B3(df_eval_b3_3, m_b3_3["exact"], m_b3_3["rel"], label="B3 Experimento 3 - Prompt v3")


-- Experimento B3-3 - Prompt v3 --

is_relevant  Acc=0.987  P=0.991  R=0.991  F1=0.991
             TP=109  FP=1  FN=1  TN=39

N2 multilabel:
  Micro F1:      0.955
  Macro F1:      0.936
  Hamming Loss:  0.0183
  Jaccard:       0.697
  Subset Acc:    0.953

  Label             P      R     F1   Sup
  -------------------------------------
  SUB_CONV      0.947  0.986  0.966    72
  SUB_CON       1.000  0.733  0.846    15
  SUB_BASE      0.962  1.000  0.980    25
  SUB_REV       0.909  1.000  0.952    10
  -------------------------------------
  Macro                       0.936

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.975  (39/40)
    card=1 (1) : 0.939  (92/98)
    card=2 (2) : 1.000  (12/12)

  Confianza: media=0.963  min=0.850  max=1.000

-- Errores N2 en relevantes · B3 Experimento 3 - Prompt v3 --
Total: 6

ID 51 | GT=['SUB_CONV'] | PRED=['SUB_REV']
  Falta: ['SUB_CONV'] | Sobra: ['SUB_REV']
  Resolución de 12/02/2025, de la Consejería de Desarrollo Sostenibl

---

##  9. Comparativa de modelos - Gemma 4B

**Problema**: Todos los experimentos anteriores usan Qwen 3.5 9B. No sabemos si el prompt es transferible a modelos más pequeños y rápidos.

**Objetivo**: Comprobar si Gemma 4 4B con el mejor prompt compacto alcanza un rendimiento comparable al 9B.

**Enfoque**: Gemma 4 4B · mejor prompt que quepa en su ventana de contexto · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [23]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider

# Cargar gemma-4-e4b-it en LM Studio antes de ejecutar
model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio")
)
agent_b3_gemma = build_agent(
    model_gemma, "v2",  # ajustar a la mejor versión compacta disponible
    output_type=ClassifierOutputB3,
    prompt_registry=PROMPT_REGISTRY_B3,
)

df_b3_gemma = await run_experiment(
    agent_b3_gemma, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b3_exp_gemma4b.csv",
    desc="B3 Gemma4B",
)
df_b3_gemma.head(3)

  Reanudando: 120/150 registros ya clasificados


B3 Gemma4B:   3%|▎         | 1/30 [01:12<34:51, 72.11s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B3 Gemma4B:   7%|▋         | 2/30 [02:00<27:01, 57.92s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


...

B3 Gemma4B:  97%|█████████▋| 29/30 [29:02<00:59, 59.64s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B3 Gemma4B: 100%|██████████| 30/30 [30:08<00:00, 60.27s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)
Tiempo: 1807.0s total  |  60.23s/item  |  30 items


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s,error
0,2,boja,"Acuerdo de 25 de febrero de 2025, de la Delega...",SUB_REV,True,SUB_REV,sector_vivienda,Inicio perdida derecho al cobro ayudas alquiler,True,acuerdo,"[""SUB_REV""]","[""sector_vivienda""]",0.95,"El acto se refiere al ""inicia procedimiento de...",115.403,NaN
1,3,boja,"Anuncio de 14 de enero de 2025, de la Direcció...",SUB_REV,True,SUB_REV,NaN,Notificacion reintegro subvencion sin sector,True,notificación,"[""SUB_REV""]",[],0.98,"La descripción menciona explícitamente ""notifi...",45.495,NaN
2,4,dog,"ACUERDO de 13 de enero de 2025, del Departamen...",SUB_REV,True,SUB_REV,sector_educacion,Inicio expedientes reintegro ayudas estudio,True,acuerdo,"[""SUB_REV""]","[""sector_educacion""]",0.98,La etiqueta SUB_REV se dispara por la mención ...,42.588,NaN


In [24]:
df_b3_gemma = pd.read_csv("../results/b3_exp_gemma4b.csv")
df_eval_b3_gemma = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b3_gemma[["description","is_relevant_pred","act_type_pred","categories_pred",
                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b3_gemma = compute_metrics_B3(df_eval_b3_gemma, "B3 Gemma 4B")
print_errors_B3(df_eval_b3_gemma, m_b3_gemma["exact"], m_b3_gemma["rel"], label="B3 Gemma 4B")


-- B3 Gemma 4B --

is_relevant  Acc=0.980  P=0.974  R=1.000  F1=0.987
             TP=110  FP=3  FN=0  TN=37

N2 multilabel:
  Micro F1:      0.934
  Macro F1:      0.943
  Hamming Loss:  0.0283
  Jaccard:       0.687
  Subset Acc:    0.893

  Label             P      R     F1   Sup
  -------------------------------------
  SUB_CONV      0.857  1.000  0.923    72
  SUB_CON       0.882  1.000  0.938    15
  SUB_BASE      1.000  0.920  0.958    25
  SUB_REV       0.909  1.000  0.952    10
  -------------------------------------
  Macro                       0.943

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.925  (37/40)
    card=1 (1) : 0.878  (86/98)
    card=2 (2) : 0.917  (11/12)

  Confianza: media=0.977  min=0.900  max=1.000

-- Errores N2 en relevantes · B3 Gemma 4B --
Total: 13

ID 9 | GT=['SUB_REV'] | PRED=['SUB_CON', 'SUB_REV']
  Falta: [] | Sobra: ['SUB_CON']
  Anuncio de 22 de enero de 2025, de la Delegación Territorial de Fomento, Articulación del ...
  Razo

## 10. Tabla resumen - Comparativa de experimentos B3

In [25]:
experimentos_cfg_B3 = [
    ("B3 Exp1 - Baseline V1", "Qwen 3.5 9B", "Zero-shot", "../results/b3_exp1_baseline_qwen9b.csv"),
    ("B3 Exp2 - Prompt V2",   "Qwen 3.5 9B", "Zero-shot", "../results/b3_exp2_promptv2_qwen9b.csv"),
    ("B3 Exp3 - Prompt V3",   "Qwen 3.5 9B", "Few-shot",  "../results/b3_exp3_promptv3_qwen9b.csv"),
    ("B3 Exp4 - Gemma 4B",    "Gemma 4 4B",  "Zero-shot", "../results/b3_exp_gemma4b.csv"),
]

rows_b3 = []
for nombre, modelo, config, path in experimentos_cfg_B3:
    if not Path(path).exists():
        print(f"  [SKIP] {nombre}: aún sin resultados ({path})")
        continue
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
        df_r[["description","is_relevant_pred","categories_pred","confidence","duration_s"]],
        on="description", how="left"
    )
    m = compute_metrics_B3(df_e, verbose=False)
    rows_b3.append({
        "Experimento":     nombre,
        "Modelo":          modelo,
        "Config":          config,
        "is_rel_f1":       m["is_rel_f1"],
        "micro_f1":        m["micro_f1"],
        "macro_f1":        m["macro_f1"],
        "hamming_loss":    m["hamming_loss"],
        "jaccard_samples": m["jaccard_samples"],
        "subset_accuracy": m["subset_accuracy"],
        "mean_duration_s": round(df_r["duration_s"].mean(), 2) if "duration_s" in df_r.columns else None,
        "total_duration_s": round(df_r["duration_s"].sum(), 0) if "duration_s" in df_r.columns else None,
    })

df_summary_b3 = pd.DataFrame(rows_b3)
if not df_summary_b3.empty:
    df_display_b3 = df_summary_b3.copy()
    df_display_b3.columns = [
        "Experimento", "Modelo", "Config",
        "is_rel F1", "Micro F1", "Macro F1", "Hamming", "Jaccard", "Subset Acc",
        "s/item", "Total (s)",
    ]
    display(df_display_b3.set_index("Experimento"))

,Modelo,Config,is_rel F1,Micro F1,Macro F1,Hamming,Jaccard,Subset Acc,s/item,Total (s)
Experimento,,,,,,,,,,
B3 Exp1 - Baseline V1,Qwen 3.5 9B,Zero-shot,0.9817,0.7857,0.7586,0.1000,0.5367,0.6800,62.24,9336.0
B3 Exp2 - Prompt V2,Qwen 3.5 9B,Zero-shot,0.9863,0.9231,0.8915,0.0317,0.6667,0.9067,63.30,9431.0
B3 Exp3 - Prompt V3,Qwen 3.5 9B,Few-shot,0.9909,0.9551,0.9362,0.0183,0.6967,0.9533,63.92,9588.0
B3 Exp4 - Gemma 4B,Gemma 4 4B,Zero-shot,0.9865,0.9339,0.9428,0.0283,0.6867,0.8933,183.35,27502.0
